# Libraries

In [1]:
import pandas as pd
import numpy as np
import datetime as dt
import logging
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import seaborn as sns
import talib as ta

from torch import optim
from torch.utils.data import DataLoader, Dataset, TensorDataset, random_split
from tqdm import tqdm

from numpy.lib.stride_tricks import sliding_window_view
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from torch.optim import AdamW
from utils import SklearnWrapper, verify_scaling
from utils.paths import CHECKPOINTS_DIR
from pypfopt import risk_models, expected_returns, plotting, EfficientFrontier

In [2]:
from config import *
from entities import *
from strategies import *
from datasets import *
from engine import Engine
from models import DiffusionTransformer, Diffusion

# Setup

In [3]:
logging.basicConfig(level=logging.DEBUG)
logging.getLogger('matplotlib').setLevel(logging.WARNING)

In [4]:
cfg = TrainConfig(
    epochs=2000
)
window_size = cfg.window_size
device = cfg.device
batch_size = cfg.batch_size
epochs = cfg.epochs

ddpm = {
    'timesteps': int(1000),
    'beta_start': 0.0001,
    'beta_end': 0.02
}

ddpm_transformer = {
    'n_features': 1,
    'n_cond': 10,
    'window_size': 60,
    'd_model': 128,
    'nhead': 4,
    'num_layers': 3,
    'dim_feedforward': 512,
    'dropout': 0.1
}

# Data [N, W, A, F]
**[N, T, A, F]** means: 
* **N**: Num of Window or Num of Batch
* **W**: Window
* **A**: Assets
* **F**: Features or Channels

In [5]:
symbols = ['AAPL', 'TSLA', 'MSFT', 'NVDA', 'GOOGL', 'AMZN', 'GOOG', 'META', 'AVGO', 'ORCL', 'CRM', 'ADBE', 'AMD', 'CSCO']
freq = "1d"

# Basket
basket = Basket(symbols=symbols)
basket.load_all_assets(freq=freq)

print(f"Basket data shape: {basket.data.shape}")
basket.data.head(5)

DEBUG:entities.basket:Initialized Asset Basket: ['AAPL', 'TSLA', 'MSFT', 'NVDA', 'GOOGL', 'AMZN', 'GOOG', 'META', 'AVGO', 'ORCL', 'CRM', 'ADBE', 'AMD', 'CSCO'] with 0 assets which loaded.
INFO:entities.basket:Starting batch load for 14 symbols...
DEBUG:entities.basket:Attempting to load AAPL...
DEBUG:entities.asset:Initialized Asset: AAPL with 2724 rows.
INFO:entities.basket:Successfully loaded AAPL (2724 rows).
DEBUG:entities.basket:Attempting to load TSLA...
DEBUG:entities.asset:Initialized Asset: TSLA with 2760 rows.
INFO:entities.basket:Successfully loaded TSLA (2760 rows).
DEBUG:entities.basket:Attempting to load MSFT...
DEBUG:entities.asset:Initialized Asset: MSFT with 2724 rows.
INFO:entities.basket:Successfully loaded MSFT (2724 rows).
DEBUG:entities.basket:Attempting to load NVDA...
DEBUG:entities.asset:Initialized Asset: NVDA with 2724 rows.
INFO:entities.basket:Successfully loaded NVDA (2724 rows).
DEBUG:entities.basket:Attempting to load GOOGL...
DEBUG:entities.asset:Initia

Basket data shape: (2760, 70)


AAPL                                                \
                Close       High        Low       Open       Volume   
Date                                                                  
2015-01-02  24.261047  24.729270  23.821672  24.718174  212818400.0   
2015-01-05  23.577574  24.110150  23.391173  24.030263  257142000.0   
2015-01-06  23.579794  23.839424  23.218085  23.641928  263188400.0   
2015-01-07  23.910433  24.010290  23.677430  23.788384  160423600.0   
2015-01-08  24.829119  24.886815  24.121236  24.238848  237458000.0   

                 TSLA                                             ...   AMD  \
                Close       High        Low       Open    Volume  ... Close   
Date                                                              ...         
2015-01-02  14.620667  14.883333  14.217333  14.858000  71466000  ...  2.67   
2015-01-05  14.006000  14.433333  13.810667  14.303333  80527500  ...  2.66   
2015-01-06  14.085333  14.280000  13.614000  14.004000  93928500  ...  2.63   
2015-01-07  14.063333  14.318667  13.985333  14.223333  44526000  ...  2.58   
2015-01-08  14.041333  14.253333  14.000667  14.187333  51637500  ...  2.61   

                                               CSCO                        \
            High   Low  Open      Volume      Close       High        Low   
Date                                                                        
2015-01-02  2.67  2.67  2.67         0.0  19.815605  20.181631  19.650534   
2015-01-05  2.70  2.64  2.67   8878200.0  19.420874  19.700776  19.377812   
2015-01-06  2.66  2.55  2.65  13912500.0  19.413698  19.865848  19.406522   
2015-01-07  2.65  2.54  2.63  12377600.0  19.593126  19.664896  19.363463   
2015-01-08  2.65  2.56  2.59  11136600.0  19.743843  20.160107  19.715135   

                                   
                 Open      Volume  
Date                               
2015-01-02  19.995029  22926500.0  
2015-01-05  19.607475  29460600.0  
2015-01-06  19.478291  47297600.0  
2015-01-07  19.478295  27570800.0  
2015-01-08  19.765374  40907000.0  

[5 rows x 70 columns]

## Features/Channels ($F$)
1. Find Joint Distribution $F_{\text{date\ A}} \cap F_{\text{date\ B}}$ with intersection
2. Select $F$ to norm as Return values

In [6]:
targets = ["Close"]
features = basket.get_unique_features()
print(f"Features:\t{features}\nTargets:\t{targets}")

Features:	['Close', 'High', 'Low', 'Open', 'Volume']
Targets:	['Close']


In [7]:
print(f"Basket data shape before Joint: {basket.data.shape}")

joint_strategy = IntersectionStrategy()
basket.align(joint_strategy)

print(f"Basket data shape after Joint: {basket.data.shape}")

INFO:strategies.concrete:Aligned: 14 orig -> 14 clean assets -> 1464 rows
DEBUG:entities.basket:Aligned data shape: (1464, 70)
INFO:entities.basket:Assets updated in-place to aligned index (Length: 1464)


Basket data shape before Joint: (2760, 70)
Basket data shape after Joint: (1464, 70)


In [8]:
basket.to_returns(features=targets, log=True, keep=False)
targets = basket.get_keyword_features("Returns")
features = basket.get_unique_features()

print(f"Features:\t{features}\nTargets:\t{targets}")
basket.data.head(5)

DEBUG:entities.asset:AAPL converted to Returns (log=True)
DEBUG:entities.asset:TSLA converted to Returns (log=True)
DEBUG:entities.asset:MSFT converted to Returns (log=True)
DEBUG:entities.asset:NVDA converted to Returns (log=True)
DEBUG:entities.asset:GOOGL converted to Returns (log=True)
DEBUG:entities.asset:AMZN converted to Returns (log=True)
DEBUG:entities.asset:GOOG converted to Returns (log=True)
DEBUG:entities.asset:META converted to Returns (log=True)
DEBUG:entities.asset:AVGO converted to Returns (log=True)
DEBUG:entities.asset:ORCL converted to Returns (log=True)
DEBUG:entities.asset:CRM converted to Returns (log=True)
DEBUG:entities.asset:ADBE converted to Returns (log=True)
DEBUG:entities.asset:AMD converted to Returns (log=True)
DEBUG:entities.asset:CSCO converted to Returns (log=True)


Features:	['Close_Log_Returns', 'High', 'Low', 'Open', 'Volume']
Targets:	{'Close_Log_Returns'}


AAPL                                                     \
                 High        Low       Open     Volume Close_Log_Returns   
Date                                                                       
2020-01-03  72.594055  71.608685  71.765667  146322800         -0.009770   
2020-01-06  72.444344  70.703034  70.954210  118387200          0.007937   
2020-01-07  72.671341  71.845369  72.415337  108872000         -0.004715   
2020-01-08  73.526310  71.768094  71.768094  132079200          0.015958   
2020-01-09  74.972955  73.951358  74.202527  170108400          0.021018   

                 TSLA                                                     ...  \
                 High        Low       Open     Volume Close_Log_Returns  ...   
Date                                                                      ...   
2020-01-03  30.266666  29.128000  29.366667  266677500          0.029203  ...   
2020-01-06  30.104000  29.333332  29.364668  151995000          0.019072  ...   
2020-01-07  31.441999  30.224001  30.760000  268231500          0.038067  ...   
2020-01-08  33.232666  31.215334  31.580000  467164500          0.048033  ...   
2020-01-09  33.253334  31.524668  33.139999  426606000         -0.022189  ...   

                  AMD                                                    \
                 High        Low       Open    Volume Close_Log_Returns   
Date                                                                      
2020-01-03  49.389999  47.540001  48.029999  73127400         -0.010236   
2020-01-06  48.860001  47.860001  48.020000  47934900         -0.004330   
2020-01-07  49.389999  48.040001  49.349998  58061400         -0.002897   
2020-01-08  48.299999  47.139999  47.849998  53767000         -0.008743   
2020-01-09  49.959999  48.389999  48.939999  76512800          0.023555   

                 CSCO                                                    
                 High        Low       Open    Volume Close_Log_Returns  
Date                                                                     
2020-01-03  40.453623  39.899004  40.260347  15577400         -0.016450  
2020-01-06  40.184715  39.504044  39.613288  22183600          0.003563  
2020-01-07  40.100678  39.579670  40.100678  16501900         -0.006507  
2020-01-08  40.159507  39.335982  39.470435  25175900          0.000632  
2020-01-09  40.235133  39.554462  40.159503  18203600         -0.004218  

[5 rows x 70 columns]

### Add Indicators as Features ($F$)

In [9]:
# Indicator
time_prd = 20
fast_prd, slow_prd, signal_prd = 12, 26, 9

for symbol, asset in basket.assets.items():
    df = asset.data 
    
    for target in targets:
        s = df[target]
        
        df[f"SMA_{time_prd} {target}"] = ta.SMA(s, timeperiod=time_prd)
        df[f"EMA_{time_prd} {target}"] = ta.EMA(s, timeperiod=time_prd)
        df[f"RSI_{time_prd} {target}"] = ta.RSI(s, timeperiod=time_prd)
        
        macd, signal, hist = ta.MACD(s, fastperiod=fast_prd, slowperiod=slow_prd, signalperiod=signal_prd)
        df[f"MACD {target}"] = macd
        df[f"MACD_Sig {target}"] = signal
        df[f"MACD_Hist {target}"] = hist

print(f"Basket data shape: {basket.data.shape}")
basket.data.head(5)

Basket data shape: (1463, 154)


AAPL                                                     \
                 High        Low       Open     Volume Close_Log_Returns   
Date                                                                       
2020-01-03  72.594055  71.608685  71.765667  146322800         -0.009770   
2020-01-06  72.444344  70.703034  70.954210  118387200          0.007937   
2020-01-07  72.671341  71.845369  72.415337  108872000         -0.004715   
2020-01-08  73.526310  71.768094  71.768094  132079200          0.015958   
2020-01-09  74.972955  73.951358  74.202527  170108400          0.021018   

                                                              \
           SMA_20 Close_Log_Returns EMA_20 Close_Log_Returns   
Date                                                           
2020-01-03                      NaN                      NaN   
2020-01-06                      NaN                      NaN   
2020-01-07                      NaN                      NaN   
2020-01-08                      NaN                      NaN   
2020-01-09                      NaN                      NaN   

                                                            \
           RSI_20 Close_Log_Returns MACD Close_Log_Returns   
Date                                                         
2020-01-03                      NaN                    NaN   
2020-01-06                      NaN                    NaN   
2020-01-07                      NaN                    NaN   
2020-01-08                      NaN                    NaN   
2020-01-09                      NaN                    NaN   

                                       ...       CSCO                       \
           MACD_Sig Close_Log_Returns  ...        Low       Open    Volume   
Date                                   ...                                   
2020-01-03                        NaN  ...  39.899004  40.260347  15577400   
2020-01-06                        NaN  ...  39.504044  39.613288  22183600   
2020-01-07                        NaN  ...  39.579670  40.100678  16501900   
2020-01-08                        NaN  ...  39.335982  39.470435  25175900   
2020-01-09                        NaN  ...  39.554462  40.159503  18203600   

                                                       \
           Close_Log_Returns SMA_20 Close_Log_Returns   
Date                                                    
2020-01-03         -0.016450                      NaN   
2020-01-06          0.003563                      NaN   
2020-01-07         -0.006507                      NaN   
2020-01-08          0.000632                      NaN   
2020-01-09         -0.004218                      NaN   

                                                              \
           EMA_20 Close_Log_Returns RSI_20 Close_Log_Returns   
Date                                                           
2020-01-03                      NaN                      NaN   
2020-01-06                      NaN                      NaN   
2020-01-07                      NaN                      NaN   
2020-01-08                      NaN                      NaN   
2020-01-09                      NaN                      NaN   

                                                              \
           MACD Close_Log_Returns MACD_Sig Close_Log_Returns   
Date                                                           
2020-01-03                    NaN                        NaN   
2020-01-06                    NaN                        NaN   
2020-01-07                    NaN                        NaN   
2020-01-08                    NaN                        NaN   
2020-01-09                    NaN                        NaN   

                                        
           MACD_Hist Close_Log_Returns  
Date                                    
2020-01-03                         NaN  
2020-01-06                         NaN  
2020-01-07                         NaN  
2020-01-08                         NaN  
202

In [10]:
basket.align(joint_strategy)

print(f"Basket data shape: {basket.data.shape}")
basket.data.head(5)

INFO:strategies.concrete:Aligned: 14 orig -> 14 clean assets -> 1430 rows
DEBUG:entities.basket:Aligned data shape: (1430, 154)
INFO:entities.basket:Assets updated in-place to aligned index (Length: 1430)


Basket data shape: (1430, 154)


AAPL                                                     \
                 High        Low       Open     Volume Close_Log_Returns   
Date                                                                       
2020-02-21  77.576610  75.167846  77.133587  129554000         -0.022895   
2020-02-24  73.637844  70.018657  71.962610  222195200         -0.048666   
2020-02-25  73.238425  69.268208  72.855932  230673600         -0.034459   
2020-02-26  72.112718  69.357773  69.365035  198054800          0.015739   
2020-02-27  69.236734  66.079924  68.050512  320605600         -0.067603   

                                                              \
           SMA_20 Close_Log_Returns EMA_20 Close_Log_Returns   
Date                                                           
2020-02-21                -0.000859                -0.001142   
2020-02-24                -0.003148                -0.005668   
2020-02-25                -0.003379                -0.008410   
2020-02-26                -0.003986                -0.006110   
2020-02-27                -0.008402                -0.011966   

                                                            \
           RSI_20 Close_Log_Returns MACD Close_Log_Returns   
Date                                                         
2020-02-21                47.685660              -0.003591   
2020-02-24                44.698556              -0.006871   
2020-02-25                46.638298              -0.008229   
2020-02-26                52.796367              -0.005195   
2020-02-27                43.935380              -0.009407   

                                       ...       CSCO                       \
           MACD_Sig Close_Log_Returns  ...        Low       Open    Volume   
Date                                   ...                                   
2020-02-21                  -0.001783  ...  38.697334  39.268761  20028700   
2020-02-24                  -0.002801  ...  36.974636  37.814969  35295900   
2020-02-25                  -0.003886  ...  35.352796  37.016655  48024700   
2020-02-26                  -0.004148  ...  35.243558  36.109103  38513200   
2020-02-27                  -0.005200  ...  33.621711  34.688934  51442900   

                                                       \
           Close_Log_Returns SMA_20 Close_Log_Returns   
Date                                                    
2020-02-21         -0.011809                -0.002834   
2020-02-24         -0.050953                -0.005228   
2020-02-25         -0.029054                -0.005248   
2020-02-26         -0.013663                -0.006246   
2020-02-27         -0.051593                -0.008067   

                                                              \
           EMA_20 Close_Log_Returns RSI_20 Close_Log_Returns   
Date                                                           
2020-02-21                -0.002368                49.539877   
2020-02-24                -0.006995                44.596763   
2020-02-25                -0.009096                47.671511   
2020-02-26                -0.009531                49.735314   
2020-02-27                -0.013537                45.119161   

                                                              \
           MACD Close_Log_Returns MACD_Sig Close_Log_Returns   
Date                                                           
2020-02-21              -0.001908                  -0.001762   
2020-02-24              -0.005450                  -0.002500   
2020-02-25              -0.006416                  -0.003283   
2020-02-26              -0.005872                  -0.003801   
2020-02-27              -0.008404                  -0.004721   

                                        
           MACD_Hist Close_Log_Returns  
Date                                    
2020-02-21                   -0.000145  
2020-02-24                   -0.002950  
2020-02-25                   -0.003133  
2020-02-26                   -0.002071  
202

### Filter only Target Features ($F_{target} $)

In [11]:
targets = basket.get_keyword_features("Returns")
print(f"Targets: {targets}")


for symbol, asset in basket.assets.items():
    mask = asset.data.columns.isin(targets)
    asset.data = asset.data.loc[:, mask]

print(f"Basket shape: {basket.data.shape}")
basket.data.head(5)

Targets: {'SMA_20 Close_Log_Returns', 'EMA_20 Close_Log_Returns', 'MACD_Hist Close_Log_Returns', 'MACD Close_Log_Returns', 'MACD_Sig Close_Log_Returns', 'Close_Log_Returns', 'RSI_20 Close_Log_Returns'}
Basket shape: (1430, 98)


AAPL                           \
           Close_Log_Returns SMA_20 Close_Log_Returns   
Date                                                    
2020-02-21         -0.022895                -0.000859   
2020-02-24         -0.048666                -0.003148   
2020-02-25         -0.034459                -0.003379   
2020-02-26          0.015739                -0.003986   
2020-02-27         -0.067603                -0.008402   

                                                              \
           EMA_20 Close_Log_Returns RSI_20 Close_Log_Returns   
Date                                                           
2020-02-21                -0.001142                47.685660   
2020-02-24                -0.005668                44.698556   
2020-02-25                -0.008410                46.638298   
2020-02-26                -0.006110                52.796367   
2020-02-27                -0.011966                43.935380   

                                                              \
           MACD Close_Log_Returns MACD_Sig Close_Log_Returns   
Date                                                           
2020-02-21              -0.003591                  -0.001783   
2020-02-24              -0.006871                  -0.002801   
2020-02-25              -0.008229                  -0.003886   
2020-02-26              -0.005195                  -0.004148   
2020-02-27              -0.009407                  -0.005200   

                                                    TSLA  \
           MACD_Hist Close_Log_Returns Close_Log_Returns   
Date                                                       
2020-02-21                   -0.001808          0.001766   
2020-02-24                   -0.004070         -0.077524   
2020-02-25                   -0.004343         -0.041482   
2020-02-26                   -0.001047         -0.026745   
2020-02-27                   -0.004207         -0.137133   

                                                              ...  \
           SMA_20 Close_Log_Returns EMA_20 Close_Log_Returns  ...   
Date                                                          ...   
2020-02-21                 0.022701                 0.020150  ...   
2020-02-24                 0.019474                 0.010847  ...   
2020-02-25                 0.018005                 0.005864  ...   
2020-02-26                 0.015879                 0.002758  ...   
2020-02-27                 0.007794                -0.010565  ...   

                              AMD                             \
           MACD Close_Log_Returns MACD_Sig Close_Log_Returns   
Date                                                           
2020-02-21              -0.005277                   0.000718   
2020-02-24              -0.010997                  -0.001625   
2020-02-25              -0.011426                  -0.003585   
2020-02-26              -0.009208                  -0.004710   
2020-02-27              -0.013302                  -0.006428   

                                                    CSCO  \
           MACD_Hist Close_Log_Returns Close_Log_Returns   
Date                                                       
2020-02-21                   -0.005995         -0.011809   
2020-02-24                   -0.009372         -0.050953   
2020-02-25                   -0.007841         -0.029054   
2020-02-26                   -0.004498         -0.013663   
2020-02-27                   -0.006874         -0.051593   

                                                              \
           SMA_20 Close_Log_Returns EMA_20 Close_Log_Returns   
Date                                                           
2020-02-21                -0.002834                -0.002368   
2020-02-24                -0.005228                -0.006995   
2020-02-25                -0.005248                -0.009096   
2020-02-26                -0.006246                -0.009531   
2020-02-27                -0.008067                -0.013537   



## Dataset & Dataloader

In [12]:
n_obs = len(basket.data)
n_assets = basket.data.columns.levels[0].size
n_features = basket.data.columns.levels[1].size

print(n_obs, n_assets, n_features)

basket_np = basket.data.values.reshape(n_obs, n_assets, n_features)
basket_np.shape

1430 14 7


(1430, 14, 7)

### Ratio Dataset

In [13]:
ratios = [0.8, 0.1, 0.1]
total_count = len(basket_np)
train_count = int(total_count * ratios[0])
val_count = int(total_count * ratios[1])
test_count = total_count - train_count - val_count

print(f"Ratios DS\nTrain:\t{train_count}\nVal:\t{val_count}\nTest:\t{test_count}\nTotal:\t{total_count}")

Ratios DS
Train:	1144
Val:	143
Test:	143
Total:	1430


In [14]:
end_val = train_count + val_count

# Ratios
train_part = basket_np[:train_count]
val_part = basket_np[train_count:end_val]
test_part = basket_np[end_val:]

print(f"Train: {train_part.shape}\nVal: {val_part.shape}\nTest:{test_part.shape}")

Train: (1144, 14, 7)
Val: (143, 14, 7)
Test:(143, 14, 7)


### Scale Dataset

In [15]:
def scale(part: np.ndarray, scaler) -> np.ndarray:
    T, A, F = part.shape
    part_2d = part.reshape(-1, F)

    # print(f"2D Part: {part_2d.shape}")

    scaled_part = scaler.transform(part_2d).reshape(T, A, F)
    return scaled_part.astype(np.float32)

In [16]:
def inspect_scaled_data(data, name):
    if isinstance(data, torch.Tensor):
        data = data.detach().cpu().numpy()

    _min = np.min(data)
    _max = np.max(data)
    _mean = np.mean(data)
    _std = np.std(data)
    
    print(f"--- Inspecting: {name} ---")
    print("-" * 36)
    print(f"Shape: {data.shape}")
    print(f"Min:   {_min:.4f}")
    print(f"Max:   {_max:.4f}")
    print(f"Mean:  {_mean:.4f}")
    print(f"Std:   {_std:.4f}")
    print("-" * 36)
    return _min, _max, _mean, _std

In [17]:
scaler = MinMaxScaler(feature_range=(-1, 1))
# scaler = StandardScaler()

# Require 2D Numpy Array
T, A, F = train_part.shape
scaler.fit(train_part.reshape(-1, F))

scaled_train_part = scale(train_part, scaler)
scaled_val_part = scale(val_part, scaler)
scaled_test_part = scale(test_part, scaler)

inspect_scaled_data(scaled_train_part, "Scaled Train Part")
inspect_scaled_data(scaled_val_part, "Scaled Val Part")
inspect_scaled_data(scaled_test_part, "Scaled Test Part")
print(f"Train:\t{scaled_train_part.shape}\nVal:\t{scaled_val_part.shape}\nTest:\t{scaled_test_part.shape}")

--- Inspecting: Scaled Train Part ---
------------------------------------
Shape: (1144, 14, 7)
Min:   -1.0000
Max:   1.0000
Mean:  0.1377
Std:   0.1432
------------------------------------
--- Inspecting: Scaled Val Part ---
------------------------------------
Shape: (143, 14, 7)
Min:   -0.7688
Max:   1.0350
Mean:  0.1300
Std:   0.1383
------------------------------------
--- Inspecting: Scaled Test Part ---
------------------------------------
Shape: (143, 14, 7)
Min:   -0.6135
Max:   1.2813
Mean:  0.1548
Std:   0.1447
------------------------------------
Train:	(1144, 14, 7)
Val:	(143, 14, 7)
Test:	(143, 14, 7)


### Dataloader

In [18]:
train_ds = MarketDataset(scaled_train_part, window_size=window_size)
val_ds = MarketDataset(scaled_val_part, window_size=window_size)
test_ds = MarketDataset(scaled_test_part, window_size=window_size)

print(f"Num of Windows\nTrain DS: {len(train_ds)}, Val Ds: {len(val_ds)}, Test DS: {len(test_ds)}\n")
print(f"A sample shape from Train DS\n\tx: {train_ds[0]['x'].shape},\n\tx_cond: {train_ds[0]['x_cond'].shape}")

Num of Windows
Train DS: 1081, Val Ds: 80, Test DS: 80

A sample shape from Train DS
	x: (64, 14, 1),
	x_cond: (64, 14, 6)


In [19]:
train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False)

batch = next(iter(train_loader))
print(len(train_loader))
print(batch["x"].shape)
print(batch["x_cond"].shape)

34
torch.Size([32, 64, 14, 1])
torch.Size([32, 64, 14, 6])


# Model, Engine
Use *Condition DDPM* 

In [20]:
# n_window mean batch size
n_window, window, n_assets, n_features = batch["x"].shape
n_window, window, n_assets, n_conds = batch["x_cond"].shape
print(n_assets, n_features, n_conds)

input_channels = n_assets * n_features
cond_channels = n_assets * n_conds
print(input_channels, cond_channels)

14 1 6
14 84


In [21]:
model = DiffusionTransformer(
    n_features=input_channels,
    n_cond=cond_channels,        
    window_size=window_size,             
    d_model=ddpm_transformer['d_model'],                
    nhead=ddpm_transformer['nhead'],
    num_layers=ddpm_transformer['num_layers'],
    dim_feedforward=ddpm_transformer['dim_feedforward'],
    dropout=ddpm_transformer['dropout']
).to(device)

In [22]:
diffusion = Diffusion(model, timesteps=ddpm['timesteps'], beta_start=ddpm['beta_start'], beta_end=ddpm['beta_end']).to(device)

In [23]:
optimizer = AdamW(model.parameters(), lr=cfg.optimizer.lr, weight_decay=cfg.optimizer.weight_decay)

In [24]:
for epoch in range(epochs):
    diffusion.train()
    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{epochs}")
    
    for batch in pbar:
        # x: [32, 64, 14, 1]
        x_raw = batch["x"].to(device).float()
        
        # x_cond: [32, 64, 14, 6]
        cond_raw = batch["x_cond"].to(device).float()
        
        batch_size = x_raw.shape[0]

        # print(f"x_raw: {x_raw.shape}, cond_raw: {cond_raw.shape}, batch_size: {batch_size}")
        
        # RESHAPE  (Flatten Assets)
        # x: [32, 64, 14, 1] -> [32, 64, 14]
        x_flat = x_raw.view(batch_size, 64, -1) 
        
        # cond: [32, 64, 14, 6] -> [32, 64, 14*6] -> [32, 64, 84] 
        cond_flat = cond_raw.view(batch_size, 64, -1)

        # print(f"x_flat: {x_flat.shape}, cond_flat: {cond_flat.shape}")
        
        optimizer.zero_grad()
        loss = diffusion(x_flat, cond_flat)
        
        loss.backward()
        optimizer.step()
        
    print(f"Epoch {epoch} Loss: {loss.item()}")
    if (epoch + 1) % 10 == 0:
        torch.save(diffusion.state_dict(), f"checkpoints/v8_ckpt_{epoch+1}.pt")

Epoch 1/2000: 100%|██████████| 34/34 [00:03<00:00,  9.10it/s]


Epoch 0 Loss: 0.9452962875366211


Epoch 2/2000: 100%|██████████| 34/34 [00:00<00:00, 64.28it/s]


Epoch 1 Loss: 0.8296117186546326


Epoch 3/2000: 100%|██████████| 34/34 [00:00<00:00, 64.86it/s]


...

Epoch 1997 Loss: 0.004435619805008173


Epoch 1999/2000: 100%|██████████| 34/34 [00:00<00:00, 64.03it/s]


Epoch 1998 Loss: 0.0008509450126439333


Epoch 2000/2000: 100%|██████████| 34/34 [00:00<00:00, 64.06it/s]


Epoch 1999 Loss: 0.0009411214850842953


In [25]:
batch = next(iter(test_loader))
x_real_raw = batch["x"].to(device).float()      # [32, 64, 14, 1]
cond_raw = batch["x_cond"].to(device).float()   # [32, 64, 14, 6]
B, W, A, F = x_real_raw.shape

# Mask (Logic: 1=known, 0=predict)
mask = torch.ones_like(x_real_raw)
mask[:, -10:, :, :] = 0  # latest 10 Assets 

# Reshape Flat
# x_real: [32, 64, 14, 1] -> [32, 64, 14]
x_start_flat = x_real_raw.view(B, W, -1)

# cond: [32, 64, 14, 6] -> [32, 64, 84]
cond_flat = cond_raw.view(B, W, -1)

# mask: [32, 64, 14, 1] -> [32, 64, 14]
mask_flat = mask.view(B, W, -1)

print(f"x_start_flat: {x_start_flat.shape}\ncond_flat: {cond_flat.shape}\nmask_flat: {mask_flat.shape}")

# Inpaint
diffusion.eval()
with torch.no_grad():
    inpainted_flat = diffusion.sample_inpaint(
        x_cond=cond_flat,
        x_start=x_start_flat,
        mask=mask_flat
    )

# Reshape
# [32, 64, 14] -> [32, 64, 14, 1]
inpainted_final = inpainted_flat.view(B, W, A, F)

print("Inpaint Finished!")

x_start_flat: torch.Size([32, 64, 14])
cond_flat: torch.Size([32, 64, 84])
mask_flat: torch.Size([32, 64, 14])
Inpaint Finished!


In [26]:
inpainted_final.shape

torch.Size([32, 64, 14, 1])